# Encyclopedic-VQA — Project overview

Answer questions about the entity in an image. Three settings, in increasing order of contribution:

| | Setting | What it is |
|---|---|---|
| **A** | No-RAG | VLM answers from image + question only |
| **B** | Standard RAG | retrieve a Wikipedia article by image, feed its text as context — the **solid baseline to beat** |
| **C** | **Agentic RAG** | our main contribution: an agent that iterates retrieval/reasoning — must beat the best B |

This repo currently builds and tunes **B** (best retrieval strategy). **C** is the next phase.

## Components

| Role | Model / asset | Notes |
|---|---|---|
| VLM (answerer) | Qwen2.5-VL-3B-Instruct | bf16 on GPU |
| Visual encoder | EVA-CLIP-8B | fp16, image + text embeddings |
| Image processor | openai/clip-vit-large-patch14 | 224×224 |
| Knowledge base | encyclopedic_kb_wiki.json (16 GB) | ~2.0M articles, url → sections |
| Image index | FAISS knn.index (8.9 GB) | ~1.66M vectors, IndexFlatIP (cosine) |
| Test set | encyclopedic_test_subset.json | 1000 examples |

## Notebook layout

- `00_overview` — this file (framing + components).
- `01_dataset` — test set exploration.
- `02_comparison` — final cross-strategy comparison (A / B* / C).
- `strategies/` — one notebook per strategy, with detail + its own results + pros/cons.

**Strategies for B (baseline) — BEM accuracy:**

| ID | Strategy | Overall |
|---|---|---|
| A | no-RAG (VLM only) | 0.245 |
| B1 | top-1 + first 3 sections | 0.278 |
| B2 | top-1 + all sections (no rerank) | 0.295 |
| B3 | top-5 + CLIP rerank | 0.243 |
| B4 | top-5 + cross-encoder rerank | **0.360** |

Best so far: **B4** (cross-encoder rerank). Note B3 (CLIP rerank) *hurts* — worse than no rerank at all.

## Engineering fixes (shared by all RAG strategies)

- **CUDA OOM (16 GB GPU):** EVA-CLIP-8B (~16 GB fp16) + Qwen do not fit on 16 GB → 45 GB GPU (L40S/A40) on `boost_usr_prod` (>24 GB VRAM policy).
- **Host RAM:** the 16 GB KB is loaded whole into a dict (~84 GB RSS) → `--mem=128G`. Structural fix: on-disk KB (SQLite).
- **device-side assert in encode_image:** non-persistent `position_ids` buffers left uninitialized under meta-device / low_cpu_mem_usage loading → re-materialize as `arange` after load.
- **Resume + separate outputs:** predictions keyed by `unique_id`; each strategy writes its own `predictions_*` / `results_*` file.

In [ ]:
import json, os

def show(path, label):
    if not os.path.exists(path):
        print('%-20s (pending)' % label); return
    r = json.load(open(path))
    print('%-20s overall %.4f (n=%d)' % (label, r['accuracy_overall'], r['num_examples']))

show('../outputs/results.json', 'A: no-RAG')
show('../outputs/results_rag.json', 'B1: top1+first3')

A: no-RAG            overall 0.2450 (n=1000)
B1: top1+first3      (pending)


## Key findings (baseline B)

- **Retrieval is the primary bottleneck.** When the correct article is retrieved, the VLM answers ~66% of the time (BEM); otherwise accuracy falls to the no-RAG level (~24%). `overall approx recall * acc_hit + (1-recall) * 0.24`. recall@5 = 28%, recall@50 = 47%.
- **top-k and top-n must be scaled together.** At low top-k, top-n is second-order (top-5: n3 0.360 approx n15 0.355). At high top-k the reranker *dilutes* and the answer paragraph slips deep in the ranking, so a larger top-n recovers it: top-50 / n5 = 0.376 -> top-50 / n20 = **0.401** (acc_hit recovers 0.53 -> 0.58). The earlier "top-k is saturated" was only true at small top-n.
- **Best baseline B so far: 0.401** (top-50 + cross-encoder + top-n 20). Theoretical ceiling at recall@50 is ~0.44, so a little headroom remains in this direction; beyond that needs a better image encoder (higher recall) or an agentic pipeline (C).
- **Reranking only matters in that you must not discard good paragraphs.** Cross-encoder (0.360) approx give-all-sections (0.347) >> CLIP rerank (0.243). A weak reranker hurts; a good one is no better than dumping everything.
- **Qwen is robust to noisy context (needle-in-haystack).** A generous reranked set beats a narrow one: it finds the answer paragraph even among many wrong-article paragraphs.
- **By question type:** `multi_answer` benefits most from more paragraphs (needs coverage: 0.259 -> 0.332); `templated` is slightly hurt by the extra noise.